In [ ]:
#| default_exp infra

In [ ]:
#| hide
import os
from fastcore.test import *
from nbdev.showdoc import *
for k in ('HCLOUD_TOKEN', 'CLOUDFLARE_API_TOKEN'): os.environ.pop(k, None)

Servers from Hetzner through `vpseasy`, tunnels and DNS from Cloudflare through `cfeasy`, with this project's tokens.

`_hetzner` and `_cf` are the only two places a client is built, and every method that reaches a provider goes through one of them. Both raise `InfraError` when the library is missing or the token is unset, so a panel has something to show before any request is made. Constructing `Infra` makes no request, and neither does `status`.

No cell on this page calls a provider. The tokens are cleared above so that none can. The examples show the fields of each answer and the failures that happen before a request.

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| export
import os

In [ ]:
#| export
from contextlib import contextmanager

In [ ]:
#| export
from pullup.env import EnvStore

In [ ]:
#| export
from pullup.stack import installed

In [ ]:
#| export
class InfraError(RuntimeError):
    pass

`InfraError` is the failure a caller is meant to show a person: a missing library, an unset token, a name that does not exist. A provider or network failure keeps its own type, and `overview` reports the two differently.

In [ ]:
#| export
@contextmanager
def _with_env(values):
    "Tokens in `os.environ` for one call only: `Hetzner` reads `HCLOUD_TOKEN` at construction."
    old = {k: os.environ.get(k) for k in values}
    os.environ.update({k: v for k, v in values.items() if v})
    try: yield
    finally:
        for k, v in old.items():
            if v is None: os.environ.pop(k, None)
            else: os.environ[k] = v

`_with_env` puts values in `os.environ` for the length of a `with` block and then restores what was there before, including restoring nothing where there was nothing. `vpseasy`'s `Hetzner` reads `HCLOUD_TOKEN` from the environment when it is constructed without a token, which is why the token is placed there rather than passed.

An empty value is not set, so a key that was unset stays unset rather than becoming `''`.

In [ ]:
os.environ['PULLUP_DEMO'] = 'before'
with _with_env({'PULLUP_DEMO': 'during', 'PULLUP_ABSENT': 'x'}):
    inside = os.environ['PULLUP_DEMO'], os.environ['PULLUP_ABSENT']
inside, os.environ['PULLUP_DEMO'], 'PULLUP_ABSENT' in os.environ

(('during', 'x'), 'before', False)

The restore also happens when the block raises, which is the case that matters: constructing a client is what runs inside it.

In [ ]:
#| hide
assert 'PULLUP_ABSENT' not in os.environ, 'a key with no value before has none after'
with _with_env({'PULLUP_DEMO': ''}): test_eq(os.environ['PULLUP_DEMO'], 'before')
def _boom():
    with _with_env({'PULLUP_DEMO': 'during'}): raise RuntimeError('client refused')
test_fail(_boom, contains='client refused')
test_eq(os.environ['PULLUP_DEMO'], 'before')
del os.environ['PULLUP_DEMO']

In [ ]:
#| export
def _record(r):
    "One DNS record as a panel shows it; `tunnel` marks a CNAME to `<id>.cfargotunnel.com`."
    content = str(r.get('content') or '')
    return {'id': r.get('id') or '', 'type': r.get('type') or '', 'name': r.get('name') or '',
            'content': content, 'proxied': bool(r.get('proxied')), 'ttl': r.get('ttl'),
            'tunnel': content.endswith('.cfargotunnel.com')}

`_record` turns one DNS record from cfeasy into what a panel shows. A field the provider omitted becomes `''`, so nothing downstream has to handle `None`. `ttl` is the exception and is passed through as given, because `1` is Cloudflare's automatic and `None` is not the same answer.

`tunnel` marks a record pointing at a Cloudflare tunnel, which is a CNAME whose content is `<tunnel id>.cfargotunnel.com`. It is matched on the end of the content, so a name that merely contains that hostname is not marked.

In [ ]:
_record({'id': 'a1', 'type': 'CNAME', 'name': 'app.example.com',
         'content': '9f2b.cfargotunnel.com', 'proxied': True, 'ttl': 1})

{'id': 'a1',
 'type': 'CNAME',
 'name': 'app.example.com',
 'content': '9f2b.cfargotunnel.com',
 'proxied': True,
 'ttl': 1,
 'tunnel': True}

In [ ]:
#| hide
r = _record({'type': 'A', 'name': 'example.com', 'content': '203.0.113.7'})
test_eq(r['id'], ''); test_eq(r['proxied'], False); test_eq(r['tunnel'], False)
test_is(r['ttl'], None)
test_eq(_record({'content': 'x.cfargotunnel.com.example.com'})['tunnel'], False)
test_eq(_record({})['content'], '')

`installed` answers whether a spec is findable, which is not the same as importable. A package whose module fails to execute is reported present and then fails at the call. `_why` separates the two, so the install hint is only given when installing would help.

In [ ]:
#| export
def _why(what, pkg, e):
    "Why `pkg` is unavailable. A findable module can still fail to execute, and then it is not missing."
    if isinstance(e, ImportError): return f'{what} need {pkg}: pip install "pullup[cloud]"'
    return f'{what} need {pkg}, which is installed but does not import: {e}'

In [ ]:
#| export
class Infra:
    "Hetzner and Cloudflare, through vpseasy and cfeasy, with this project's tokens."
    HCLOUD, CF_TOKEN = 'HCLOUD_TOKEN', 'CLOUDFLARE_API_TOKEN'
    def __init__(self, env=None):
        self.env = env or EnvStore()
    def token(self, key):
        try: return self.env.get(key) or os.environ.get(key, '')
        except Exception: return os.environ.get(key, '')
    def status(self):
        return {'vpseasy': installed('vpseasy'), 'cfeasy': installed('cfeasy'),
                'hcloud_token': bool(self.token(self.HCLOUD)),
                'cf_token': bool(self.token(self.CF_TOKEN)),
                'keys': [self.HCLOUD, self.CF_TOKEN]}
    def _hetzner(self):
        try: from vpseasy.core import Hetzner
        except Exception as e: raise InfraError(_why('servers', 'vpseasy', e)) from e
        if not self.token(self.HCLOUD):
            raise InfraError('HCLOUD_TOKEN is not set — add it in Workflows → Environment')
        with _with_env({self.HCLOUD: self.token(self.HCLOUD)}): return Hetzner()
    def _cf(self):
        try: from cfeasy.core import CF
        except Exception as e: raise InfraError(_why('tunnels and DNS', 'cfeasy', e)) from e
        token = self.token(self.CF_TOKEN)
        if not token: raise InfraError('CLOUDFLARE_API_TOKEN is not set — add it in Workflows → Environment')
        return CF(token=token)
    def servers(self):
        "Every Hetzner server on this token, as vpseasy reports them."
        return list(self._hetzner().servers())
    def delete_server(self, name):
        "Delete one server, named. Irreversible, and the caller is expected to have asked."
        name = str(name or '').strip()
        if not name: raise InfraError('name the server to delete')
        known = {s['name'] for s in self.servers()}
        if name not in known: raise InfraError(f'no server called {name}')
        self._hetzner().delete(name)
        return {'deleted': name}
    def keys(self): return list(self._hetzner().keys())
    def tunnels(self):
        "Live Cloudflare tunnels, trimmed to what a panel can show."
        rows = [{'id': t.get('id') or '', 'name': t.get('name') or '',
                 'status': t.get('status') or '', 'created_at': str(t.get('created_at') or ''),
                 'deleted_at': str(t.get('deleted_at') or ''),
                 'connections': len(t.get('connections') or [])} for t in self._cf().tunnels()]
        return [r for r in rows if not r['deleted_at']]
    def delete_tunnel(self, tunnel_id):
        tunnel_id = str(tunnel_id or '').strip()
        if not tunnel_id: raise InfraError('name the tunnel to delete')
        self._cf().delete_tunnel(tunnel_id)
        return {'deleted': tunnel_id}
    def zones(self):
        return [{'id': z.get('id') or '', 'name': z.get('name') or '', 'status': z.get('status') or ''}
                for z in self._cf().zones()]
    def records(self, zone):
        "DNS records for one zone, named or by id."
        cf = self._cf()
        zone = str(zone or '').strip()
        if not zone: raise InfraError('choose a zone')
        zid = zone if len(zone) == 32 and '.' not in zone else cf.zone_id(zone)
        return sorted((_record(r) for r in cf.dns_records(zid)), key=lambda r: (r['name'], r['type']))
    def verify(self):
        "cfeasy's own token check, so a permissions problem is named before a deploy hits it."
        return self._cf().verify()
    def overview(self):
        "Everything at once as `{rows, error}` each, so Cloudflare being down cannot hide the servers."
        out = {'status': self.status()}
        for key, fn in (('servers', self.servers), ('tunnels', self.tunnels), ('zones', self.zones)):
            try: out[key] = {'rows': fn(), 'error': ''}
            except Exception as e:
                out[key] = {'rows': [], 'error': str(e) if isinstance(e, InfraError) else f'{type(e).__name__}: {e}'}
        return out

A package that is present but broken reports as available and then fails at the call. On Python 3.10 and 3.11 `vpseasy` 0.0.18 is exactly that: `vpseasy/core.py` spells a nested f-string the way only 3.12 parses, so importing it raises `SyntaxError`. The guard catches that as well as `ImportError`, so what reaches the caller is an `InfraError` naming the cause rather than a traceback from inside somebody else's module.

In [ ]:
#| hide
import sys, tempfile
from pathlib import Path
brk = Path(tempfile.mkdtemp())/'brokenmod'; brk.mkdir()
(brk/'__init__.py').write_text('from .core import *\n')
(brk/'core.py').write_text('x = (1\n')                         # an unclosed paren: a SyntaxError
sys.path.insert(0, str(brk.parent))
test_eq(installed('brokenmod'), True)                          # findable, and still not importable
test_fail(lambda: __import__('brokenmod'), contains='')
try: __import__('brokenmod')
except Exception as e: msg = _why('servers', 'brokenmod', e)
assert 'does not import' in msg and 'pip install' not in msg, msg
assert 'pip install "pullup[cloud]"' in _why('servers', 'nope', ImportError('no module'))
sys.path.remove(str(brk.parent))

`Infra` resolves each token itself: the project's `EnvStore` first, then this process's environment, then `''`. A store that raises is not a failure, and the environment answers instead. Any object with a `get(key)` is accepted, which is how a caller supplies tokens without a keychain.

In [ ]:
infra = Infra(env={'HCLOUD_TOKEN': 'hc-demo'})
infra.token('HCLOUD_TOKEN'), infra.token('CLOUDFLARE_API_TOKEN')

('hc-demo', '')

In [ ]:
#| hide
class _Locked:
    def get(self, key): raise RuntimeError('keychain locked')
os.environ['CLOUDFLARE_API_TOKEN'] = 'cf-from-env'
test_eq(Infra(env=_Locked()).token('CLOUDFLARE_API_TOKEN'), 'cf-from-env')
test_eq(Infra(env=_Locked()).token('HCLOUD_TOKEN'), '')
del os.environ['CLOUDFLARE_API_TOKEN']

`status` is what a panel asks before offering anything: whether each library is importable and whether each token has a value. It reports no token values and makes no request. `keys` names the two environment variables it read.

`vpseasy` being importable is not the same as importing: `installed` reads the spec without running the module.

In [ ]:
Infra(env={'HCLOUD_TOKEN': 'hc-demo'}).status()

{'vpseasy': True,
 'cfeasy': True,
 'hcloud_token': True,
 'cf_token': False,
 'keys': ['HCLOUD_TOKEN', 'CLOUDFLARE_API_TOKEN']}

In [ ]:
#| hide
s = Infra(env={'HCLOUD_TOKEN': 'hc-demo'}).status()
test_eq(s['hcloud_token'], True); test_eq(s['cf_token'], False)
test_eq(s['keys'], ['HCLOUD_TOKEN', 'CLOUDFLARE_API_TOKEN'])
assert 'hc-demo' not in str(s), 'a status report carries no token values'

What each of the listing methods asks for, and what comes back:

- `servers` lists the Hetzner servers on the token, as vpseasy reports them: `{name, ip, status}` each, with `ip` the public IPv4.
- `keys` lists the SSH keys on the same token: `{name, fingerprint}` each.
- `delete_server` lists the servers first and refuses a name that is not among them, then deletes that server. It answers `{'deleted': name}`. Deleting a server is not reversible, and nothing here asks for confirmation.
- `tunnels` lists the Cloudflare tunnels of the first account on the token, trimmed to `{id, name, status, created_at, connections}`, where `connections` is a count rather than the connection records. A tunnel with a `deleted_at` is dropped rather than shown.
- `delete_tunnel` deletes one tunnel by id and answers `{'deleted': id}`.
- `zones` lists the zones on the token as `{id, name, status}`.
- `records` lists the DNS records of one zone through `_record`, sorted by name then type. `zone` is either a zone id or a domain name: 32 characters with no dot is taken as an id, and anything else is looked up by name, which costs one request more.
- `verify` is cfeasy's own token check, `{'result': bool}` and an `err` key when it failed. It reads a zone and a tunnel list, so it names a permissions problem before a deploy hits it.

Both deletions check their argument before anything is built, so an empty name fails without a request.

In [ ]:
test_fail(lambda: Infra(env={'HCLOUD_TOKEN': 'hc-demo'}).delete_server(' '), contains='name the server')
test_fail(lambda: Infra(env={'HCLOUD_TOKEN': 'hc-demo'}).delete_tunnel(''), contains='name the tunnel')

`overview` answers with every section at once, each as `{rows, error}`. Cloudflare being down cannot hide the servers, and a missing `cfeasy` cannot either. An `InfraError` is reported by its message alone. Anything else keeps its type name in front. A `KeyError` out of a provider's SDK is not a message to hand a person on its own. `status` is included unwrapped, since it cannot fail.

In [ ]:
o = Infra(env={'HCLOUD_TOKEN': '', 'CLOUDFLARE_API_TOKEN': ''}).overview()
o['zones']

{'rows': [],
 'error': 'CLOUDFLARE_API_TOKEN is not set — add it in Workflows → Environment'}

In [ ]:
#| hide
for k in ('servers', 'tunnels', 'zones'):
    test_eq(o[k]['rows'], [])
    assert o[k]['error'], f'{k} reported neither rows nor a reason'
test_eq(o['status']['cf_token'], False)
test_eq(o['zones']['error'], o['tunnels']['error'])